# AfriVoices — SOUMISSION avec DECOUPAGE EN FENETRES (clips >20s)

29% du test depasse 20s (mas 36%, kln 32%, swa 42%). Le modele n'a jamais vu de tels
audios a l'entrainement (filtre <20s). Ici : clips <=20s transcrits normalement ; clips
>20s decoupes en fenetres de 15s (recouvrement 2s), chaque fenetre transcrite, puis
textes recolles en dedupliquant la zone de recouvrement.

**Seul bloc a regler : section 2** (MODEL_NAME, LM, alpha, beta, TAG).
Reprise auto tous les 10 parquets. Compare a ton meilleur (0.40283).

## 1. Dependances (redemarrer le runtime apres la 1ere execution)

In [ ]:
import importlib
need = []
for m in ["kenlm","pyctcdecode"]:
    try: importlib.import_module(m)
    except ImportError: need.append(m)
if need:
    import subprocess
    subprocess.run(["pip","install","-q","pyctcdecode"], check=False)
    subprocess.run(["pip","install","-q","https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("REDEMARRE le runtime (Execution > Redemarrer la session), puis relance cette cellule.")
else:
    print("kenlm + pyctcdecode disponibles - continue en section 2")

## 2. CONFIG - le seul bloc a modifier

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"      # modele a soumettre (dossier sur le Drive)
LM_SUBDIR  = "lm_v2"
ALPHA, BETA = 0.7, 0.5
TAG = "v9_2_fenetres"
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
PARTIEL = f"{BASE}/submission_{TAG}_partiel.csv"
FINAL   = f"{BASE}/submission_{TAG}.csv"
assert os.path.isdir(MODEL), f"modele absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modele: {MODEL_NAME} | LM: {LM_SUBDIR} | alpha={ALPHA} beta={BETA}")
print(f"sortie: {FINAL}")

## 3. Rechargement : modele + labels + decodeurs + decode_robuste

In [ ]:
import torch, io, base64, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok:
        labels.append("")
    elif t in ("|", " "):
        labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}:
        n += 1
        labels.append("\u2047" * n)
    else:
        labels.append(t)
assert labels.count("") == 1 and labels.count(" ") == 1

def decode_robuste(b):
    if not b:
        raise ValueError("vide")
    try:
        arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try:
            arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                try:
                    tf.write(base64.b64decode(b))
                except Exception:
                    tf.write(b)
                p = tf.name
            arr, sr = librosa.load(p, sr=16000, mono=True)
            os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)
    if sr != 16000:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top=50000):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f:
            c.update(line.split())
    return [w for w, _ in c.most_common(top)]

decoders = {}
for lang in ["swa", "kik", "luo", "kln", "mas", "som"]:
    decoders[lang] = build_ctcdecoder(
        labels,
        kenlm_model_path=f"{LM}/{lang}.bin",
        unigrams=unigrams(lang),
        alpha=ALPHA, beta=BETA,
    )
print("pret :", MODEL_NAME, "+", LM_SUBDIR, f"(alpha={ALPHA}, beta={BETA})")

## 4. Donnees de test (Drive si present, sinon telechargement HF)

In [ ]:
import glob
TEST_CANDIDATES = [f"{BASE}/test", "/content/test_data"]
parquets = []
for td in TEST_CANDIDATES:
    parquets = sorted(glob.glob(f"{td}/**/*.parquet", recursive=True))
    if parquets:
        TEST_DIR = td
        break
if not parquets:
    print("test absent localement -> telechargement HF...")
    from huggingface_hub import snapshot_download
    TEST_DIR = "/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
    parquets = sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")
assert parquets, "aucun parquet test trouve" 

## 5. Inference avec decoupage en fenetres (clips >20s)

In [ ]:
import pandas as pd, time
from tqdm.auto import tqdm

MAX_SEC_BATCH = 160.0
WIN_SEC   = 15.0
OVER_SEC  = 2.0
SEUIL_SEC = 20.0

def fenetres(arr, sr=16000):
    w = int(WIN_SEC * sr)
    o = int(OVER_SEC * sr)
    step = w - o
    segs = []
    i = 0
    while i < len(arr):
        segs.append(arr[i:i + w])
        if i + w >= len(arr):
            break
        i += step
    return segs

def recolle(textes):
    if not textes:
        return ""
    out = textes[0].split()
    for t in textes[1:]:
        mots = t.split()
        if not mots:
            continue
        best = 0
        for k in range(1, min(6, len(out), len(mots)) + 1):
            if out[-k:] == mots[:k]:
                best = k
        out += mots[best:]
    return " ".join(out)

def logits_batch(arrs):
    """Retourne les logits pour chaque array (meme ordre), batching dynamique par duree."""
    order = sorted(range(len(arrs)), key=lambda k: len(arrs[k]))
    out = [None] * len(arrs)
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and (j - i + 1) * (len(arrs[order[j]]) / 16000.0) <= MAX_SEC_BATCH:
            j += 1
        idxs = order[i:j]
        batch = [arrs[k] for k in idxs]
        feats = processor.feature_extractor(batch, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            lg = model(**{k: v.to(device) for k, v in feats.items()}).logits.cpu().numpy()
        for bi, k in enumerate(idxs):
            out[k] = lg[bi]
        i = j
    return out

done = {}
if os.path.exists(PARTIEL):
    dfp = pd.read_csv(PARTIEL)
    done = dict(zip(dfp["id"].astype(str), zip(dfp["language"], dfp["transcription"])))
    print("reprise :", len(done), "deja transcrits")

rows = [{"id": k, "language": v[0], "transcription": v[1]} for k, v in done.items()]
t0 = time.time()

for pi, pq in enumerate(tqdm(parquets, desc="Inference", unit="pq")):
    df = pd.read_parquet(pq)
    buckets = {}   # lang -> [(rid, arr)]
    for _, r in df.iterrows():
        rid = str(r["id"])
        if rid in done:
            continue
        lang = r.get("language")
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        try:
            arr = decode_robuste(b)
        except Exception:
            rows.append({"id": rid, "language": lang, "transcription": "_"})
            continue
        buckets.setdefault(lang, []).append((rid, arr))

    for lang, its in buckets.items():
        dec = decoders.get(lang) or next(iter(decoders.values()))
        seg_arrs = []
        seg_map = []   # (rid, fenetre_idx)
        for rid, arr in its:
            if len(arr) / 16000.0 <= SEUIL_SEC:
                seg_map.append((rid, 0))
                seg_arrs.append(arr)
            else:
                for fi, seg in enumerate(fenetres(arr)):
                    seg_map.append((rid, fi))
                    seg_arrs.append(seg)
        lgs = logits_batch(seg_arrs)
        par_rid = {}
        for k, (rid, fi) in enumerate(seg_map):
            par_rid.setdefault(rid, []).append((fi, dec.decode(lgs[k])))
        for rid, segs in par_rid.items():
            segs.sort(key=lambda x: x[0])
            txt = recolle([t for _, t in segs]) if len(segs) > 1 else segs[0][1]
            rows.append({"id": rid, "language": lang, "transcription": txt})

    if (pi + 1) % 10 == 0 or pi == len(parquets) - 1:
        pd.DataFrame(rows).to_csv(PARTIEL, index=False)
        print(f"parquet {pi + 1}/{len(parquets)} | {len(rows)} transcrits | {(time.time() - t0) / 60:.0f} min")

print("inference terminee :", len(rows))

## 6. Assemblage final + asserts + CSV

In [ ]:
import pandas as pd
sub = pd.DataFrame(rows)
sub["transcription"] = (sub["transcription"].astype(str)
                        .str.replace(r"[\r\n]+", " ", regex=True).str.strip())
sub.loc[sub["transcription"].isin(["", "nan", "None"]), "transcription"] = "_"
sub["id"] = sub["id"].astype(str)
sub = sub.drop_duplicates(subset="id", keep="first")[["id", "language", "transcription"]]

assert list(sub.columns) == ["id", "language", "transcription"]
assert len(sub) == 41733, f"attendu 41733, obtenu {len(sub)}"
assert sub["id"].is_unique
assert sub["language"].notna().all()
assert (sub["transcription"].str.len() > 0).all()

sub.to_csv(FINAL, index=False)
print("soumission ecrite ->", FINAL)
print(sub["language"].value_counts())
sub.head(8)